[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 04](README.md)

# MPI colectivas, datatypes y topologías

**Tema:** 04 · **Sesiones:** 19, 20 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Qué patrón colectivo expresa la comunicación y cómo cambia su costo con procesos y datos?


## Resultados de aprendizaje

- Seleccionar broadcast, scatter/gather, reduce/allreduce.
- Modelar costo con latencia y ancho de banda.
- Calcular vecinos de una topología cartesiana.


## Modelo conceptual

Las colectivas deben invocarse en orden compatible por todos los procesos del comunicador.

Un datatype derivado describe layout; no convierte automáticamente tipos ni corrige extensiones erróneas.

Las topologías asocian estructura lógica y pueden facilitar mapeo, sin garantizar colocación física.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "04"
NOTEBOOK = "04_mpi/02_colectivas_topologias.ipynb"
assert (ROOT / "curso" / "notebooks" / "04_mpi" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Costo de colectivas

Se compara un modelo lineal con uno arbóreo para broadcast.


In [ ]:
import math
latency_us, bandwidth_gbs, bytes_ = 2.0, 12.0, 8_000_000
transfer_us = bytes_ / (bandwidth_gbs * 1e9) * 1e6
assert transfer_us > latency_us
for p in (2, 4, 8, 16, 32):
    linear = (p-1) * (latency_us + transfer_us)
    tree = math.ceil(math.log2(p)) * (latency_us + transfer_us)
    assert tree <= linear
    print(f"p={p:2} lineal={linear:9.1f}us árbol={tree:9.1f}us")


**Interpretación.** El modelo orienta la hipótesis; la biblioteca puede segmentar, usar árboles distintos y adaptar el algoritmo al tamaño.


## Vecinos cartesianos

Se enumeran coordenadas y vecinos sin periodicidad en una malla 3×4.


In [ ]:
rows, cols = 3, 4
def rank(r, c): return r*cols+c if 0 <= r < rows and 0 <= c < cols else None
assert rank(0, 0) == 0 and rank(2, 3) == 11 and rank(-1, 0) is None
for r in range(rows):
    for c in range(cols):
        neighbors = {"N": rank(r-1,c), "S": rank(r+1,c), "W": rank(r,c-1), "E": rank(r,c+1)}
        print(rank(r,c), (r,c), neighbors)


**Interpretación.** En MPI, `MPI_Cart_shift` obtiene vecinos de acuerdo con dimensiones, periodicidad y posible reordenamiento.


## Práctica reproducible

1. Reemplazar una secuencia manual por la colectiva equivalente.
2. Verificar counts y desplazamientos para tamaños irregulares.
3. Medir colectiva por tamaño de mensaje y número de procesos.


## Errores frecuentes

- Invocar colectivas en órdenes distintos.
- Suponer que reduce entrega resultado a todos.
- Crear datatype sin revisar extent.

## Criterios de aceptación

- Orden colectivo compatible.
- Counts, tipos y buffers válidos en cada rango.
- Modelo de costo contrastado con datos.


## Referencias y material relacionado

- [Ejemplos MPI](../../../mpi/)
- [Planeación MPI](../../../docs/PLANEACION_CURSO.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 04](README.md)
